# Tennessee Eastman Fault 1: raw context and behavioral objects

This notebook asks a narrow question: **what information does the FeatureGraph oscillation representation add to, or remove from, reactor-pressure fault detection?**

The comparison uses the same object boundaries for every representation:

1. **Raw context:** ordinary statistics from the unsmoothed pressure samples inside each oscillation interval.
2. **FeatureGraph:** intrinsic oscillation properties only.
3. **Combined:** raw context and intrinsic oscillation properties together.

Fault 1 runs 1–3 are used for training and runs 4–5 are held out. The fault-free trajectory is an external negative control. Splitting by complete simulation run prevents neighboring objects from leaking between training and evaluation.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import featuregraph as fg

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)


In [ ]:
SIGNAL = "reactor_pressure"
GROUP = ["fault_number", "simulation_run"]

FAULT_START = 600
RESPONSE_END = 1200

TRAIN_RUNS = (1, 2, 3)
TEST_RUNS = (4, 5)

RAW_FEATURES = [
    "raw_mean",
    "raw_std",
    "raw_min",
    "raw_max",
    "raw_range",
    "raw_start",
    "raw_end",
    "raw_delta",
    "raw_abs_delta",
]

FEATUREGRAPH_FEATURES = [
    "rise_duration",
    "fall_duration",
    "duration",
    "period",
    "amplitude",
    "rising_mean_rate",
    "falling_mean_rate",
    "peak_rise_rate",
    "peak_fall_rate",
    "temporal_symmetry",
]

REPRESENTATIONS = {
    "raw_context": RAW_FEATURES,
    "featuregraph": FEATUREGRAPH_FEATURES,
    "combined": RAW_FEATURES + FEATUREGRAPH_FEATURES,
}


## Construct comparable examples

The raw and FeatureGraph representations below share each oscillation's start and end boundaries. This isolates the effect of the **object properties** from the effect of segmentation.

Two targets are retained:

- `post_injection`: every object available after the known injection index. This is the conventional persistent fault label.
- `transient_response`: objects overlapping the visibly disturbed pressure interval. This tests whether the representation captures the closed-loop response itself.

The second target is descriptive, not an independent ground-truth fault annotation.


In [ ]:
def construct_examples(
    *,
    dataset,
    fault_number,
    simulation_run,
):
    observations = fg.datasets.eastman(
        dataset=dataset,
        fault_number=fault_number,
        simulation_run=simulation_run,
    ).reset_index(drop=True)

    constructor = fg.oscillation.Oscillation(
        signals=SIGNAL,
        group=GROUP,
        smooth_signal=True,
        smooth_window=20,
        diff_lag=10,
    )
    features = constructor.fit_transform(observations)
    behavior_objects = constructor.summarize(
        features,
        signal=SIGNAL,
    )
    objects = behavior_objects.table.copy()
    objects = objects.loc[objects["is_complete"]].reset_index(drop=True)

    raw_rows = []
    for row in objects.itertuples(index=False):
        start = int(row.start_index)
        end = int(row.end_index)
        segment = observations[SIGNAL].iloc[start : end + 1]

        raw_rows.append(
            {
                "raw_mean": segment.mean(),
                "raw_std": segment.std(ddof=0),
                "raw_min": segment.min(),
                "raw_max": segment.max(),
                "raw_range": segment.max() - segment.min(),
                "raw_start": segment.iloc[0],
                "raw_end": segment.iloc[-1],
                "raw_delta": segment.iloc[-1] - segment.iloc[0],
                "raw_abs_delta": abs(segment.iloc[-1] - segment.iloc[0]),
            }
        )

    examples = pd.concat(
        [objects, pd.DataFrame(raw_rows)],
        axis=1,
    )
    examples["dataset"] = dataset
    examples["condition"] = (
        "fault_free" if fault_number == 0 else f"fault_{fault_number}"
    )

    if fault_number == 0:
        examples["post_injection"] = 0
        examples["transient_response"] = 0
        examples["regime"] = "fault_free"
    else:
        examples["post_injection"] = (
            examples["end_index"] >= FAULT_START
        ).astype(int)
        examples["transient_response"] = (
            (examples["end_index"] >= FAULT_START)
            & (examples["start_index"] <= RESPONSE_END)
        ).astype(int)
        examples["regime"] = np.select(
            [
                examples["end_index"] < FAULT_START,
                examples["start_index"] <= RESPONSE_END,
            ],
            ["pre_injection", "transient_response"],
            default="post_response",
        )

    return observations, features, behavior_objects, examples


In [ ]:
fault_runs = {}
example_frames = []

for run in (*TRAIN_RUNS, *TEST_RUNS):
    print(f"Constructing Fault 1, run {run}...")
    result = construct_examples(
        dataset="faulty_training",
        fault_number=1,
        simulation_run=run,
    )
    fault_runs[run] = result
    example_frames.append(result[-1])

print("Constructing fault-free control...")
normal_result = construct_examples(
    dataset="faultfree_training",
    fault_number=0,
    simulation_run=1,
)
normal_observations, _, normal_objects, normal_examples = normal_result

fault_examples = pd.concat(example_frames, ignore_index=True)

print("\nFault objects by run and regime:")
display(
    fault_examples.groupby(["simulation_run", "regime"])
    .size()
    .rename("objects")
    .unstack(fill_value=0)
)

print("\nFault-free control objects:", len(normal_examples))


## Inspect the raw trajectories

The vertical lines mark the hypothesized injection index and the end of the visibly disturbed response window. These positions must be checked across all runs before being treated as final annotations.


In [ ]:
fig, axes = plt.subplots(
    len(fault_runs) + 1,
    1,
    figsize=(14, 12),
    sharex=True,
    sharey=True,
)

axes[0].plot(
    normal_observations[SIGNAL].to_numpy(),
    linewidth=1,
    label="fault-free",
)
axes[0].set_title("Fault-free control")
axes[0].legend(loc="upper right")

for axis, (run, result) in zip(axes[1:], fault_runs.items()):
    observations = result[0]
    axis.plot(
        observations[SIGNAL].to_numpy(),
        linewidth=1,
        label=f"Fault 1, run {run}",
    )
    axis.axvline(
        FAULT_START,
        color="tab:red",
        linestyle="--",
        linewidth=1,
        label="injection",
    )
    axis.axvline(
        RESPONSE_END,
        color="tab:orange",
        linestyle=":",
        linewidth=1,
        label="response-window end",
    )
    axis.legend(loc="upper right")

axes[-1].set_xlabel("sample index")
fig.supylabel("reactor pressure")
fig.tight_layout()


## Regime-level object changes

This table is a manual audit before modeling. Large differences in amplitude, duration, or rates should be visible here if FeatureGraph captures the transient response.


In [ ]:
regime_summary = (
    fault_examples.groupby("regime")[
        [
            "amplitude",
            "duration",
            "period",
            "rising_mean_rate",
            "falling_mean_rate",
            "raw_mean",
            "raw_std",
            "raw_range",
        ]
    ]
    .median()
    .round(4)
)

regime_summary


In [ ]:
def evaluate_representation(
    *,
    target,
    representation,
    feature_columns,
):
    train = fault_examples[
        fault_examples["simulation_run"].isin(TRAIN_RUNS)
    ].copy()
    test = fault_examples[
        fault_examples["simulation_run"].isin(TEST_RUNS)
    ].copy()

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=0,
        ),
    )
    model.fit(
        train[feature_columns],
        train[target],
    )

    test_probability = model.predict_proba(
        test[feature_columns]
    )[:, 1]
    test_prediction = (test_probability >= 0.5).astype(int)

    control_probability = model.predict_proba(
        normal_examples[feature_columns]
    )[:, 1]
    control_prediction = (control_probability >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        test[target],
        test_prediction,
        labels=[0, 1],
    ).ravel()

    detection_delays = []
    for run in TEST_RUNS:
        run_rows = test[
            test["simulation_run"].eq(run)
        ].copy()
        run_rows["probability"] = test_probability[
            test["simulation_run"].eq(run).to_numpy()
        ]
        detected = run_rows[
            (run_rows["end_index"] >= FAULT_START)
            & (run_rows["probability"] >= 0.5)
        ].sort_values("end_index")

        delay = (
            np.nan
            if detected.empty
            else max(
                0,
                float(detected.iloc[0]["end_index"]) - FAULT_START,
            )
        )
        detection_delays.append(delay)

    predictions = test[
        [
            "fault_number",
            "simulation_run",
            "oscillation_id",
            "start_index",
            "end_index",
            "regime",
            target,
        ]
    ].copy()
    predictions["target"] = target
    predictions["representation"] = representation
    predictions["probability"] = test_probability
    predictions["prediction"] = test_prediction

    result = {
        "target": target,
        "representation": representation,
        "feature_count": len(feature_columns),
        "train_objects": len(train),
        "test_objects": len(test),
        "positive_test_objects": int(test[target].sum()),
        "roc_auc": roc_auc_score(test[target], test_probability),
        "average_precision": average_precision_score(
            test[target],
            test_probability,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            test[target],
            test_prediction,
        ),
        "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "faultfree_false_positive_rate": control_prediction.mean(),
        "median_detection_delay_samples": np.nanmedian(
            detection_delays
        ),
    }
    return result, predictions, model


In [ ]:
evaluation_rows = []
prediction_frames = []
models = {}

for target in ("post_injection", "transient_response"):
    for representation, feature_columns in REPRESENTATIONS.items():
        result, predictions, model = evaluate_representation(
            target=target,
            representation=representation,
            feature_columns=feature_columns,
        )
        evaluation_rows.append(result)
        prediction_frames.append(predictions)
        models[(target, representation)] = model

evaluation = pd.DataFrame(evaluation_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

evaluation.round(4)


## Detection probabilities over time

A useful representation should rise near the disturbance without repeatedly alarming during ordinary fault-free operation. Remember that an object-level decision becomes available only at its `end_index`.


In [ ]:
target = "transient_response"
target_predictions = predictions[
    predictions["target"].eq(target)
]

fig, axes = plt.subplots(
    len(TEST_RUNS),
    1,
    figsize=(14, 6),
    sharex=True,
    sharey=True,
)

for axis, run in zip(axes, TEST_RUNS):
    run_rows = target_predictions[
        target_predictions["simulation_run"].eq(run)
    ]

    for representation in REPRESENTATIONS:
        series = run_rows[
            run_rows["representation"].eq(representation)
        ].sort_values("end_index")
        axis.plot(
            series["end_index"],
            series["probability"],
            marker="o",
            markersize=3,
            linewidth=1,
            label=representation,
        )

    axis.axvline(FAULT_START, color="tab:red", linestyle="--")
    axis.axvline(RESPONSE_END, color="tab:orange", linestyle=":")
    axis.axhline(0.5, color="0.5", linestyle="--", linewidth=1)
    axis.set_title(f"Held-out Fault 1 run {run}")
    axis.set_ylabel("fault-response probability")
    axis.legend(loc="upper right")

axes[-1].set_xlabel("oscillation end index")
fig.tight_layout()


## Interpretation guide

- If **raw context** wins, the absolute operating level or raw within-object variation carries most of the detectable signal.
- If **FeatureGraph** wins, the temporal shape of the pressure response is distinctive even without absolute pressure.
- If **combined** wins, the behavioral properties complement operating context.
- A large gap between `transient_response` and `post_injection` performance means the controller restores pressure observability even though the injected disturbance remains active.
- A high fault-free false-positive rate means the representation confuses ordinary oscillations with the fault response.

This experiment evaluates **reactor-pressure detection of Fault 1**, not diagnosis among all Tennessee Eastman fault types.


## Preserve the generated evidence

Running the next cell creates reviewable CSV and JSON artifacts. These files should be inspected before they are committed.


In [ ]:
OUTPUT_DIR = Path("artifacts/tep/fault_1_representation_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fault_examples.to_csv(
    OUTPUT_DIR / "fault_1_objects.csv",
    index=False,
)
normal_examples.to_csv(
    OUTPUT_DIR / "faultfree_objects.csv",
    index=False,
)
evaluation.to_csv(
    OUTPUT_DIR / "evaluation.csv",
    index=False,
)
predictions.to_csv(
    OUTPUT_DIR / "heldout_predictions.csv",
    index=False,
)

manifest = {
    "signal": SIGNAL,
    "fault_number": 1,
    "fault_start_index": FAULT_START,
    "response_end_index": RESPONSE_END,
    "train_runs": list(TRAIN_RUNS),
    "test_runs": list(TEST_RUNS),
    "representations": REPRESENTATIONS,
    "notes": [
        "The response-end index is an exploratory visual annotation.",
        "Raw-context features use FeatureGraph object boundaries.",
        "The fault-free trajectory is an external negative control.",
    ],
}

with (OUTPUT_DIR / "manifest.json").open("w") as file:
    json.dump(manifest, file, indent=2)

print("Artifacts written to", OUTPUT_DIR)
